# NGC 4395 and POX 52

This notebook applies the workflow of the Fairall 9 tutorial to two dwarf galaxies with low-mass AGN, NGC 4395 and POX 52. It covers

1. querying the VizieR SED photometry of both targets;
2. fitting each target with its own stellar-mass prior and a host-capture model;
3. comparing the stellar masses and AGN luminosities of the two fits.

The redshifts are fixed to their spectroscopic values. You need internet access and the DSPS SSP file `tempdata.h5` in the repository root.

In [ ]:
import numpy as np
import numpyro.distributions as dist
from astropy.table import Table

from jaxsedfit import (
    AGNConfig,
    FilterSet,
    FitConfig,
    GalaxyConfig,
    InferenceConfig,
    JAXSEDFit,
    LikelihoodConfig,
    Observation,
    PhotometryData,
    PriorConfig,
    load_filter_curves,
)
from jaxsedfit.vizier import query_vizier_sed

## 1. Targets and photometry

In [ ]:
targets = {
    "NGC 4395": {"redshift": 0.001064, "log_mass_prior": 8.5},
    "POX 52": {"redshift": 0.0218, "log_mass_prior": 9.0},
}

photometry = {}
for name in targets:
    rows, _, _ = query_vizier_sed(name, radius=5.0)
    photometry[name] = Table(rows=rows)["grahsp_filter", "flux_mjy", "err_mjy", "psf_fwhm_arcsec"]
    print(f"{name}: {len(rows)} bands: {list(photometry[name]['grahsp_filter'])}")

## 2. Fit each target

As for Fairall 9, `use_host_capture_model=True` lets the captured host light vary with the spatial resolution of each survey.

In [ ]:
def make_config(name, target, phot):
    return FitConfig(
        observation=Observation(object_id=name, redshift=target["redshift"]),
        photometry=PhotometryData(
            filter_names=list(phot["grahsp_filter"]),
            fluxes=list(phot["flux_mjy"]),
            errors=list(phot["err_mjy"]),
            psf_fwhm_arcsec=list(phot["psf_fwhm_arcsec"]),
        ),
        filters=FilterSet(curves=load_filter_curves(list(phot["grahsp_filter"]))),
        galaxy=GalaxyConfig(dsps_ssp_fn="../tempdata.h5"),
        agn=AGNConfig(agn_type=1),
        likelihood=LikelihoodConfig(use_host_capture_model=True),
        inference=InferenceConfig(method="optax+nuts", map_steps=400, learning_rate=5e-3, num_warmup=100, num_samples=50),
        prior_config=PriorConfig(
            stellar_mass=dist.Normal(target["log_mass_prior"], 1.0),
            host={"ebv_gal": dist.HalfNormal(0.15)},
            agn={"ebv_agn": dist.HalfNormal(0.15)},
        ),
    )


fitters = {}
results = {}
for name, target in targets.items():
    fitters[name] = JAXSEDFit(make_config(name, target, photometry[name]))
    results[name] = fitters[name].fit(progress_bar=False)

## 3. Results

The stellar mass comes from `result.samples` and the dereddened AGN bolometric luminosity from `predict()`, as in the Fairall 9 tutorial. Luminosities are in W. Add 7 to the logarithms for erg/s.

In [ ]:
def summary(name, values):
    lo, mid, hi = np.percentile(values, [16, 50, 84])
    return f"{name} {mid:.2f} (+{hi - mid:.2f}/-{mid - lo:.2f})"


for name, fitter in fitters.items():
    pred = fitter.predict()
    print(
        f"{name}: "
        + summary("log M*", results[name].samples["log_stellar_mass"])
        + ", "
        + summary("log L_bol,AGN (erg/s)", pred["log_agn_bol_luminosity_dereddened_fit"] + 7.0)
    )
    fitter.plot_sed(show=True, title=name)

## Next steps

- Increase `num_warmup` and `num_samples` for science fits; these short chains are for illustration.
- Set `aperture_diameter_arcsec` in `PhotometryData` when the photometric apertures are known, instead of the nominal PSF widths.
- Fit many targets at once with `jaxsedfit.fit_batch`.